In [1]:
!python --version

Python 3.11.14


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv/perturb_agent
SRC added: /home/flavio/uv/perturb_agent/src


### Defaults

In [3]:
root0 = ROOT / "data"
gdc = GDC(root0=root0)

os.listdir(root0)[:10]

['cancer',
 'reactome_summary.tsv',
 'reactome',
 'vector_store',
 'reactome_pathways.tsv',
 'TCGA',
 'gdc_programs.txt']

### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/gdc_programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.self.cancer.gov/projects'

In [7]:
prog_id = 'TCGA'
df_psi = gdc.get_primary_sites(prog_id=prog_id, force=force, verbose=verbose)

primary_sites = np.unique(df_psi.primary_site)
primary_sites[:10]

Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/TCGA/primary_site_program_TCGA.tsv'


array(['Adrenal gland',
       'Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and related structures, Other and ill-defined sites, Connective, subcutaneous and other soft tissues, Spinal cord, cranial nerves, and other parts of central nervous system, Heart, mediastinum, and pleura',
       'Bladder', 'Brain', 'Breast', 'Bronchus and lung',
       'Bronchus and lung, Heart, mediastinum, and pleura',
       'Cervix uteri, Ovary', 'Colon, Rectosigmoid junction',
       'Colon, Rectosigmoid junction, Rectum, Connective, subcutaneous and other soft tissues, Unknown'],
      dtype=object)

In [8]:
primary_site = 'Breast'
gdc.set_primary_site(primary_site = primary_site)

primary_site = gdc.primary_site
psi_id = gdc.psi_id

psi_id, primary_site

('TCGA-BRCA', 'Breast')

In [9]:
verbose=False

df_cases, df_tumor_samples, df_tummor_mut, tumor_barcode_list = gdc.get_filtered_tables(primary_site=primary_site, sample_type_term='Primary Tumor', verbose=verbose)

In [10]:
verbose=False

df_cases, df_normal_samples, df_normal_mut, normal_barcode_list = gdc.get_filtered_tables(primary_site=primary_site, sample_type_term='Solid Tissue Normal', verbose=verbose)

In [11]:
df_cases.head(2)

,primary_site,disease_type,case_id,diagnoses,psi_id,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,primary_site_norm,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac
0,Breast,Ductal and Lobular Neoplasms,c25898b4-eb33-42f4-bf3f-bf532a929e7d,"[{'primary_diagnosis': 'Lobular carcinoma, NOS...",TCGA-BRCA,lobular,unknown,"Lobular carcinoma, NOS",NaN,NaN,...,breast,ductal and lobular neoplasms,lobular carcinoma,other,epithelial,breast_lobular,ok,valid,1,0.000911
1,Breast,Ductal and Lobular Neoplasms,bb8d42d3-ad65-4d88-ae1d-f9aadfc7962d,"[{'primary_diagnosis': 'Lobular carcinoma, NOS...",TCGA-BRCA,lobular,unknown,"Lobular carcinoma, NOS",NaN,NaN,...,breast,ductal and lobular neoplasms,lobular carcinoma,other,epithelial,breast_lobular,ok,valid,1,0.000911


In [12]:
df_normal_samples.head(3)

,case_id,submitter_id,sample_id,sample_type,barcode_sample,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
0,37242f5a-25ae-4b1f-9ce6-09ce1dc92539,TCGA-E2-A15L,6369b291-fa41-4ca3-bf3b-90fcfd7b917e,Solid Tissue Normal,TCGA-E2-A15L-11A,36857cce-06d3-4eb4-9d2c-137e2be5747f,f7f1f468-a589-4593-82e4-abbf09d26fcc_wxs_gdc_r...,Aligned Reads,BAM,TCGA-BRCA,lobular,other,breast_lobular,unknown
1,37242f5a-25ae-4b1f-9ce6-09ce1dc92539,TCGA-E2-A15L,6369b291-fa41-4ca3-bf3b-90fcfd7b917e,Solid Tissue Normal,TCGA-E2-A15L-11A,9216f3c5-c7c2-4abf-aa2b-c6e4fe5cf235,HITCH_p_TCGASNP_b93_N_GenomeWideSNP_6_A03_7414...,Simple Germline Variation,TSV,TCGA-BRCA,lobular,other,breast_lobular,unknown
2,37242f5a-25ae-4b1f-9ce6-09ce1dc92539,TCGA-E2-A15L,6369b291-fa41-4ca3-bf3b-90fcfd7b917e,Solid Tissue Normal,TCGA-E2-A15L-11A,64047edb-5625-44ce-a5b8-883a47a4cd5a,66705e98-15ad-4eab-9901-7d1f6c0c2e57_noid_Red....,Masked Intensities,IDAT,TCGA-BRCA,lobular,other,breast_lobular,unknown


In [13]:
df_tumor_samples.groupby(['sample_type', 'data_format', 'data_format']).size()

sample_type    data_format  data_format
Primary Tumor  BAM          BAM             648
               BEDPE        BEDPE           424
               CEL          CEL             109
               IDAT         IDAT            218
               MAF          MAF             578
               PDF          PDF             109
               SVS          SVS             252
               TAR          TAR             100
               TSV          TSV             959
               TXT          TXT             866
               VCF          VCF            1484
dtype: int64

In [14]:
df_normal_samples.groupby(['sample_type', 'data_format', 'data_format']).size()

sample_type          data_format  data_format
Solid Tissue Normal  BAM          BAM            12
                     BEDPE        BEDPE           4
                     CEL          CEL             4
                     IDAT         IDAT            6
                     MAF          MAF             6
                     SVS          SVS             9
                     TSV          TSV            15
                     TXT          TXT            17
                     VCF          VCF             8
dtype: int64

### Cases with normal tissue

In [15]:
np.unique(df_tumor_samples.data_type)

array(['Aggregated Somatic Mutation', 'Aligned Reads',
       'Allele-specific Copy Number Segment',
       'Annotated Somatic Mutation', 'Copy Number Segment',
       'Gene Expression Quantification', 'Gene Level Copy Number',
       'Intermediate Analysis Archive',
       'Isoform Expression Quantification', 'Masked Copy Number Segment',
       'Masked Intensities', 'Masked Somatic Mutation',
       'Methylation Beta Value', 'Pathology Report',
       'Protein Expression Quantification', 'Raw Intensities',
       'Raw Simple Somatic Mutation', 'Simple Germline Variation',
       'Slide Image', 'Splice Junction Quantification',
       'Structural Rearrangement', 'Transcript Fusion',
       'miRNA Expression Quantification'], dtype=object)

In [16]:
dff_normal = df_normal_samples[df_normal_samples.data_type == 'Gene Expression Quantification']
print(len(dff_normal))

2


In [17]:
case_id_list = np.unique(dff_normal.case_id)
print(len(case_id_list))

case_id_list[:3]

2


array(['02bbb632-0f7f-439d-b8f0-c86a06237424',
       '14267783-5624-4fe5-ba81-9d67f1017474'], dtype=object)

In [18]:
dff_tumor = df_tumor_samples[df_tumor_samples.case_id.isin(case_id_list)].copy()
dff_tumor = dff_tumor[dff_tumor.data_type == 'Gene Expression Quantification']

print(len(dff_tumor))
dff_tumor.head(6)

2


,case_id,submitter_id,sample_id,sample_type,barcode_sample,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
1771,14267783-5624-4fe5-ba81-9d67f1017474,TCGA-BH-A0DP,ffb838b0-db61-4dc9-aadc-41562f677f54,Primary Tumor,TCGA-BH-A0DP-01A,e5b0c2dc-c652-40a0-bb80-d7e87830b406,8f8c9d55-303c-4099-8038-458e3e96ad4e.rna_seq.a...,Gene Expression Quantification,TSV,TCGA-BRCA,lobular,other,breast_lobular,unknown
2764,02bbb632-0f7f-439d-b8f0-c86a06237424,TCGA-AC-A2FB,7e2ed541-38ee-473f-ac5d-3cae6f5d9a56,Primary Tumor,TCGA-AC-A2FB-01A,fde3b574-b9be-4dc2-a5ae-aaab8a0bbd26,ca9d1ab5-ea78-46f4-9225-8147639d013c.rna_seq.a...,Gene Expression Quantification,TSV,TCGA-BRCA,lobular,other,breast_lobular,unknown


### Preparing to get Gene Expression files from TCGA programa

In [23]:
verbose=True
force=False

dfexp = pd.DataFrame()

for i, row in dff_normal.iterrows():
    if int(i)%10==0:
        print(i, end='')
    else:
        print(".", end='')
    case_id = row.case_id
    file_id = row.file_id

    dfexp = gdc.get_table_given_fileID(case_id=case_id, file_id=file_id, sample_type='normal', file_type='Gene Expression Quantification', verbose=verbose, force=force)

print(len(dfexp))
dfexp.head(3)
                    

.Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/Gene_Expression_Quantification_normal_for_TCGA-BRCA_case_14267783-5624-4fe5-ba81-9d67f1017474_file_934935ae-037b-451d-a696-0679b07bd401.tsv'
.Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/Gene_Expression_Quantification_normal_for_TCGA-BRCA_case_02bbb632-0f7f-439d-b8f0-c86a06237424_file_e914127b-20a0-4ef1-a614-06b697da648c.tsv'
60660


,gene_id,symbol,gene_type,unstranded,counts,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,ENSG00000000003,TSPAN6,protein_coding,5362,2763,2600,53.9024,16.6032,17.4841
1,ENSG00000000005,TNMD,protein_coding,1214,598,616,37.5048,11.5524,12.1653
2,ENSG00000000419,DPM1,protein_coding,1924,957,967,72.6863,22.3891,23.5769


In [22]:
verbose=True
force=False

dfexp = pd.DataFrame()

for i, row in dff_tumor.iterrows():
    if int(i)%10==0:
        print(i, end='')
    else:
        print(".", end='')
    case_id = row.case_id
    file_id = row.file_id

    dfexp = gdc.get_table_given_fileID(case_id=case_id, file_id=file_id, sample_type='tumor', file_type='Gene Expression Quantification', verbose=verbose, force=force)

print(len(dfexp))
dfexp.head(3)
                    

.Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/Gene_Expression_Quantification_tumor_for_TCGA-BRCA_case_14267783-5624-4fe5-ba81-9d67f1017474_file_e5b0c2dc-c652-40a0-bb80-d7e87830b406.tsv'
.Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/Gene_Expression_Quantification_tumor_for_TCGA-BRCA_case_02bbb632-0f7f-439d-b8f0-c86a06237424_file_fde3b574-b9be-4dc2-a5ae-aaab8a0bbd26.tsv'
60660


,gene_id,symbol,gene_type,unstranded,counts,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,ENSG00000000003,TSPAN6,protein_coding,4518,2308,2210,53.9018,15.6271,15.5481
1,ENSG00000000005,TNMD,protein_coding,10,3,7,0.3666,0.1063,0.1058
2,ENSG00000000419,DPM1,protein_coding,1657,841,816,74.2926,21.5388,21.4299
